# FACET Data Analysis and Occupation Design

This notebook is the first step of the revised experiment design. Instead of manually choosing occupations first, we use FACET to define a broad occupation/role vocabulary, then check how well those labels can be matched to BLS and ABS labour-statistics vocabularies.

The guiding choice is intentionally permissive: keep FACET `class1` labels that are not rare in the local single-person image subset. These labels can all be used as CLIP text prompts. The BLS/ABS matching table is used later to decide which subset is suitable for real-world labour-statistics comparison.

In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import re
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clip_bias.data import resolve_dataset_path

try:
    display
except NameError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

FACET_ROOT = resolve_dataset_path(PROJECT_ROOT, "facet")
BLS_ROOT = resolve_dataset_path(PROJECT_ROOT, "bls")
ABS_ROOT = resolve_dataset_path(PROJECT_ROOT, "abs")

FACET_ANNOTATIONS = FACET_ROOT / "annotations" / "annotations_single_person.csv"
FACET_IMAGE_DIR = FACET_ROOT / "images"
BLS_PATH = BLS_ROOT / "raw" / "cpsaat11.xlsx"
ABS_PATH = ABS_ROOT / "raw" / "EQ08.xlsx"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "facet_occupation_selection"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MIN_IMAGES = 25
CONFIG_OUT = PROJECT_ROOT / "configs" / "occupations_facet_broad.yml"

PROJECT_ROOT


## 1. FACET Data Status

The local FACET image directory has already been reduced to single-person images. Here we verify the current annotation/image alignment and inspect the class distribution that will drive occupation selection.

In [ ]:
facet = pd.read_csv(FACET_ANNOTATIONS)
available_images = {path.name for path in FACET_IMAGE_DIR.glob("*") if path.is_file()}
facet = facet[facet["filename"].isin(available_images)].copy()

class_counts = facet["class1"].value_counts().rename_axis("facet_class").reset_index(name="n_images")
class_counts["occupation"] = class_counts["facet_class"].str.replace("_", " ", regex=False)
class_counts.to_csv(OUTPUT_DIR / "facet_class_counts_on_disk.csv", index=False)

summary = pd.DataFrame(
    [
        {"metric": "single_person_images_on_disk", "value": len(facet)},
        {"metric": "unique_class1_labels", "value": class_counts["facet_class"].nunique()},
        {"metric": f"class1_labels_with_n_ge_{MIN_IMAGES}", "value": int(class_counts["n_images"].ge(MIN_IMAGES).sum())},
    ]
)
display(summary)
class_counts.head(60)

In [ ]:
plt.figure(figsize=(12, 10))
plot_df = class_counts.head(52).copy()
sns.barplot(data=plot_df, y="occupation", x="n_images", color="#4C78A8")
plt.axvline(MIN_IMAGES, color="#D62728", linestyle="--", linewidth=1.5, label=f"threshold = {MIN_IMAGES}")
plt.title("FACET single-person class distribution")
plt.xlabel("Images on disk")
plt.ylabel("FACET class1")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "facet_class_distribution.png", dpi=200)
plt.show()

## 2. Broad Occupation Candidate List

We keep every FACET `class1` label with at least `MIN_IMAGES` available images. This is deliberately broad: some labels are occupations, some are roles, and some are activities. The advantage is that the CLIP prompt vocabulary is grounded in the actual FACET data distribution.

In [ ]:
broad_candidates = class_counts[class_counts["n_images"].ge(MIN_IMAGES)].copy()
broad_candidates["category"] = "facet_frequent"
broad_candidates = broad_candidates[["occupation", "facet_class", "n_images", "category"]]
broad_candidates.to_csv(OUTPUT_DIR / f"facet_broad_occupation_candidates_min{MIN_IMAGES}.csv", index=False)

config = {
    "prompt_templates": [
        "a photo of a {occupation}",
        "a portrait of a {occupation}",
        "this person is a {occupation}",
        "a person working as a {occupation}",
    ],
    "occupations": {"facet_frequent": broad_candidates["occupation"].tolist()},
}
with open(CONFIG_OUT, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=False)

print(f"Broad candidate occupations: {len(broad_candidates)}")
print(f"Config written to: {CONFIG_OUT}")
broad_candidates

## 3. BLS and ABS Vocabulary Extraction

BLS and ABS use formal occupation taxonomies. The automatic matching below is a first-pass review tool, not a final semantic judgment. We save match strength so later analyses can separate CLIP-only prompts from labour-statistics-comparable occupations.

In [ ]:
bls_raw = pd.read_excel(BLS_PATH, sheet_name="cpsaat11", header=None)
bls_vocab = bls_raw.iloc[:, 0].dropna().astype(str).str.strip()
bls_vocab = bls_vocab[~bls_vocab.str.startswith("HOUSEHOLD DATA")]
bls_vocab = bls_vocab[~bls_vocab.str.startswith("[")]
bls_vocab = bls_vocab[~bls_vocab.isin(["Occupation", "Total, 16 years and over", "Total"])]
bls_vocab = bls_vocab.drop_duplicates().tolist()

abs_raw = pd.read_excel(ABS_PATH, sheet_name="Data 1", header=3)
abs_col = "Occupation of main job: ANZSCO (2013) v1.2"
abs_vocab = (
    abs_raw[abs_col]
    .dropna()
    .astype(str)
    .str.replace(r"^\s*\d{4}\s+", "", regex=True)
    .drop_duplicates()
    .tolist()
)

pd.DataFrame(
    [
        {"source": "FACET broad candidates", "unique_labels": len(broad_candidates)},
        {"source": "BLS detailed occupations", "unique_labels": len(bls_vocab)},
        {"source": "ABS ANZSCO unit groups", "unique_labels": len(abs_vocab)},
    ]
)

## 4. Matching Helpers

In [ ]:
def normalize_text(value: str) -> str:
    text = str(value).lower().replace("_", " ").replace("&", " and ")
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def singularize(token: str) -> str:
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("s") and not token.endswith("ss") and len(token) > 3:
        return token[:-1]
    return token

def canonical(value: str) -> str:
    return " ".join(singularize(token) for token in normalize_text(value).split())

ALIASES = {
    "lawman": ["police officer", "law enforcement officer", "protective service worker"],
    "computer user": ["computer user", "ict professional", "software and applications programmer", "computer occupation"],
    "repairman": ["repairer", "repair worker", "maintenance worker", "mechanic"],
    "boatman": ["ship and boat captain", "boat operator", "marine transport professional"],
    "craftsman": ["craft worker", "artisan", "trades worker"],
    "seller": ["retail salesperson", "sales worker", "sales assistant"],
    "guard": ["security guard", "security officer"],
    "fireman": ["firefighter", "fire fighter"],
    "disk jockey": ["disc jockey", "radio disc jockey", "announcer"],
    "ballplayer": ["athlete", "sports competitor"],
    "tennis player": ["athlete", "sports competitor"],
    "basketball player": ["athlete", "sports competitor"],
    "soccer player": ["athlete", "sports competitor"],
    "runner": ["athlete", "sports competitor"],
    "horseman": ["athlete", "sports competitor"],
    "guitarist": ["musician", "music professional"],
    "drummer": ["musician", "music professional"],
    "singer": ["musician", "singer"],
}

MANUAL_MATCHES = {
    "nurse": ("Registered nurses", "Registered Nurses"),
    "doctor": ("Other physicians", "General Practitioners and Resident Medical Officers"),
    "boatman": ("Ship and boat captains and operators", "Marine Transport Professionals"),
    "lawman": ("Police officers", "Police"),
    "fireman": ("Firefighters", "Defence Force Members, Fire Fighters and Police nfd"),
    "disk jockey": ("Broadcast announcers and radio disc jockeys", "Music Professionals"),
}

def score_match(query: str, candidate: str) -> tuple[float, str]:
    forms = [canonical(query)] + [canonical(alias) for alias in ALIASES.get(query, [])]
    candidate_canon = canonical(candidate)
    if candidate_canon in forms:
        return 1.0, "exact_or_alias_exact"
    for form in forms:
        if form and form in candidate_canon:
            return 0.92, "contains_alias"
    return max(SequenceMatcher(None, form, candidate_canon).ratio() for form in forms), "fuzzy"

def best_match(query: str, vocab: list[str]) -> tuple[str, float, str, str]:
    scored = []
    for candidate in vocab:
        score, method = score_match(query, candidate)
        scored.append((score, method, candidate))
    score, method, candidate = sorted(scored, reverse=True, key=lambda item: (item[0], item[2]))[0]
    strength = "strong" if score >= 0.90 else "possible" if score >= 0.78 else "weak"
    return candidate, round(score, 3), strength, method


## 5. Match Review

In [ ]:
match_rows = []
for row in broad_candidates.itertuples(index=False):
    query = row.occupation
    if query in MANUAL_MATCHES:
        bls_label, abs_label = MANUAL_MATCHES[query]
        bls_score = abs_score = 1.0
        bls_strength = abs_strength = "strong"
        bls_method = abs_method = "manual_reviewed"
    else:
        bls_label, bls_score, bls_strength, bls_method = best_match(query, bls_vocab)
        abs_label, abs_score, abs_strength, abs_method = best_match(query, abs_vocab)

    match_rows.append(
        {
            "occupation": query,
            "facet_class": row.facet_class,
            "n_images": int(row.n_images),
            "bls_best_match": bls_label,
            "bls_score": bls_score,
            "bls_strength": bls_strength,
            "bls_method": bls_method,
            "abs_best_match": abs_label,
            "abs_score": abs_score,
            "abs_strength": abs_strength,
            "abs_method": abs_method,
        }
    )

match_review = pd.DataFrame(match_rows)
match_review["labour_stats_usable"] = match_review["bls_strength"].isin(["strong", "possible"]) & match_review[
    "abs_strength"
].isin(["strong", "possible"])
match_review.to_csv(OUTPUT_DIR / "facet_broad_bls_abs_match_review.csv", index=False)

print(f"BLS+ABS usable matches: {match_review['labour_stats_usable'].sum()} / {len(match_review)}")
match_review

In [ ]:
plot_df = match_review.sort_values("n_images", ascending=False).copy()
plt.figure(figsize=(12, 10))
sns.barplot(data=plot_df, y="occupation", x="n_images", hue="labour_stats_usable", dodge=False, palette={True: "#2CA02C", False: "#9E9E9E"})
plt.title("FACET broad occupations and BLS/ABS match usability")
plt.xlabel("FACET images")
plt.ylabel("Occupation / role prompt")
plt.legend(title="Usable for BLS+ABS")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "facet_broad_candidates_match_usability.png", dpi=200)
plt.show()

match_review.groupby(["bls_strength", "abs_strength", "labour_stats_usable"], as_index=False).size()